# NB3 — Time Travel + MERGE Upsert

**Mục tiêu:** demo 4 demo-block tasks from slide line ~700.
Maps to deliverable bullet 3.

In [1]:
import sys, time, random
sys.path.append("/workspace/scripts")
from spark_session import get_spark
from delta.tables import DeltaTable
from pyspark.sql import functions as F

spark = get_spark("nb3_time_travel")
path = "s3a://lakehouse/customers_tt"

## 1. Build version history
v0: initial 100K customers · v1: schema add · v2: MERGE upsert 100K · v3: bad data ingest

In [2]:
v0 = spark.range(100_000).select(
    F.col("id").alias("customer_id"),
    F.lit("active").alias("status"),
    (F.col("id") % 1000).cast("int").alias("score"),
)
v0.write.format("delta").mode("overwrite").save(path)             # v0

# v1 — add column
df1 = (spark.read.format("delta").load(path)
       .withColumn("tier", F.when(F.col("score") > 800, "gold").otherwise("silver")))
df1.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path)  # v1

# v2 — MERGE upsert 100K
updates = spark.range(50_000, 150_000).select(
    F.col("id").alias("customer_id"),
    F.lit("vip").alias("status"),
    F.lit(999).alias("score"),
    F.lit("platinum").alias("tier"),
)
target = DeltaTable.forPath(spark, path)
t0 = time.time()
(target.alias("t").merge(updates.alias("s"), "t.customer_id = s.customer_id")
       .whenMatchedUpdateAll()
       .whenNotMatchedInsertAll()
       .execute())                                                # v2
print(f"MERGE 100K rows: {time.time()-t0:.2f}s")

# v3 — simulate bad data
bad = spark.range(50).select(F.col("id").alias("customer_id"),
                              F.lit(None).cast("string").alias("status"),
                              F.lit(-1).alias("score"),
                              F.lit("UNKNOWN").alias("tier"))
bad.write.format("delta").mode("append").save(path)               # v3

MERGE 100K rows: 15.28s


## 2. DESCRIBE HISTORY — audit trail

In [3]:
spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

+-------+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                                                                                                                                                                                                            

## 3. Time travel queries

In [4]:
print("v0:", spark.read.format("delta").option("versionAsOf", 0).load(path).count())
print("v1 schema:", spark.read.format("delta").option("versionAsOf", 1).load(path).columns)

v0: 100000
v1 schema: ['customer_id', 'status', 'score', 'tier']


## 4. RESTORE bad version

In [5]:
t0 = time.time()
DeltaTable.forPath(spark, path).restoreToVersion(2)
print(f"RESTORE: {time.time()-t0:.2f}s   (target < 30s)")

# Verify the bad rows are gone
bad_count = (spark.read.format("delta").load(path)
             .where("score < 0").count())
print(f"Rows with score<0 after restore: {bad_count}  (expected 0)")

RESTORE: 8.01s   (target < 30s)
Rows with score<0 after restore: 0  (expected 0)


## 5. Final history — now includes the RESTORE row

(Earlier `DESCRIBE HISTORY` ran *before* the RESTORE, so it would have
shown only 4 versions. This second pass is the one to screenshot.)

In [6]:
final = spark.sql(f"DESCRIBE HISTORY delta.`{path}`").select("version", "operation").collect()
for r in final:
    print(f"  v{r['version']:>2}  {r['operation']}")
print(f"\nTotal versions: {len(final)}  (target ≥ 5)")

  v 9  RESTORE
  v 8  WRITE
  v 7  MERGE
  v 6  WRITE
  v 5  WRITE
  v 4  RESTORE
  v 3  WRITE
  v 2  MERGE
  v 1  WRITE
  v 0  WRITE

Total versions: 10  (target ≥ 5)


## ✅ Deliverable check
- [x] DESCRIBE HISTORY shows ≥ 5 versions (incl. RESTORE itself)
- [x] MERGE 100K finished in < 60s
- [x] RESTORE finished in < 30s and removed bad rows

In [7]:
spark.stop()